# 📘 Notebook 01 — Environment Setup & Data Exploration

**Goals:**
1. Install all dependencies
2. Authenticate with HuggingFace
3. Load and explore the base model (Qwen2.5-1.5B)
4. Load and explore ILDC + NyayaAnumana datasets
5. Format data for instruction tuning and save

**Runtime:** ~15 min | **GPU:** Required

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q sentencepiece protobuf huggingface_hub
!pip install -q rouge-score bert-score nltk evaluate
!pip install -q openai pandas numpy tqdm

## 2. Setup Path & Imports

In [ ]:
import sys, os

# Add project root to path so we can import from src/
# On Kaggle: upload the src/ folder as a dataset or utility script
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

## 3. Authenticate with HuggingFace

In [ ]:
from huggingface_hub import login
from src.config import HF_TOKEN

login(HF_TOKEN)
print("✅ HuggingFace login successful")

## 4. Load the Base Model (Qwen2.5-1.5B)

In [ ]:
from src.model_utils import load_base_model

model, tokenizer = load_base_model()

# Quick model info
print(f"\nVocab size: {tokenizer.vocab_size}")
print(f"Model type: {model.config.model_type}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num layers: {model.config.num_hidden_layers}")

## 5. Quick Sanity Check — Generate Before Fine-tuning

In [ ]:
from src.model_utils import generate_judgment

# Test with a sample legal question
sample_facts = """The accused was charged under Section 302 IPC for
allegedly murdering his neighbor during a property dispute.
The prosecution presented three eyewitnesses and forensic evidence.
The defense argued it was a case of self-defense under Section 96 IPC."""

print("=== BASE MODEL (before fine-tuning) ===")
base_output = generate_judgment(model, tokenizer, sample_facts)
print(base_output)

## 6. Load Indian Legal Datasets

In [ ]:
from src.data_utils import load_legal_datasets

ildc, nyaya = load_legal_datasets()

## 7. Explore the Datasets

In [ ]:
# ILDC structure
print("=== ILDC Dataset ===")
print(f"Columns: {ildc.column_names}")
print(f"Num samples: {len(ildc)}")
print(f"\nSample entry (first 500 chars):")
sample = ildc[0]
for key, val in sample.items():
    display_val = str(val)[:500] if isinstance(val, str) else val
    print(f"  {key}: {display_val}")

In [ ]:
# NyayaAnumana structure
print("=== NyayaAnumana Dataset ===")
print(f"Columns: {nyaya.column_names}")
print(f"Num samples: {len(nyaya)}")
print(f"\nSample entry (first 500 chars):")
sample = nyaya[0]
for key, val in sample.items():
    display_val = str(val)[:500] if isinstance(val, str) else val
    print(f"  {key}: {display_val}")

## 8. Format Data for Instruction Tuning

In [ ]:
from src.data_utils import prepare_sft_dataset

dataset = prepare_sft_dataset(ildc)

# Preview a formatted sample
print("\n=== Formatted Sample Preview ===")
print(dataset['train'][0]['text'][:800])

## 9. Summary

✅ **Completed:**
- Environment set up with all dependencies
- Base model loaded (Qwen2.5-1.5B-Instruct, 4-bit quantized)
- Both datasets loaded and explored (ILDC + NyayaAnumana)
- Data formatted for instruction tuning with bilingual prompts

**Next → Notebook 02:** Generate chain-of-thought data using DeepSeek-R1